In [ ]:
import gzip
import glob
import csv
import math
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

aa=['GLYD', 'SCP', 'SCA', 'SCV', 'SCL', 'SCI', 'SCC', 'SCM', 'SCS', 'SCT', 'SCN', 'SCQ',
    'SCF', 'SCY', 'SCW', 'SCCM', 'SCYM', 'SCE', 'SCEN', 'SCD',
    'SCDN', 'SCK', 'SCKN', 'SCR', 'SCRN', 'SCHD', 'SCHE', 'SCHP']

save_dir='../plot/SUPP_monomer/'

# Load monomer/multimer rates
_rate_file = '../data/SUPP_monomer/monomer_multimer_rates_45A_9batches.dat'
_rate_df = pd.read_csv(_rate_file, sep=r'\s+', comment='#', encoding='latin-1')
monomer_rates  = dict(zip(_rate_df['SC'], _rate_df['mono_mean']))
multimer_rates = dict(zip(_rate_df['SC'], _rate_df['multi_mean']))
print(f'Loaded rates for {len(monomer_rates)} analogs from {_rate_file}')

In [ ]:
##############################################################################
# PMF mono vs total (using monomer_4.5A and total PMF data, no % scaling)
##############################################################################
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

PMF_TOTAL = '../data/pmf_data/total'
PMF_MONO = '../data/pmf_data/monomer_4.5A'
PMF_MULTI = '../data/pmf_data/multimer_4.5A'

def free_energy(pKa, charge):
    R=8.314*10**(-3) #kJ/mol.K
    T=302.3545798089109 #K
    pH=7.0
    return -charge*R*T*np.log(10)*(pKa-pH)

def get_ref_per_acid(acid):
    #source : Platzer et al. (2014)
    neutrals=['SCHP', 'SCKN', 'SCRN', 'SCDN', 'SCEN', 'SCCM', 'SCYM']
    pKas=[6.45, 10.34, 13.9, 3.86, 4.34, 8.49, 9.76]
    charges=[1, -1, -1, 1, 1, -1, -1]
    if acid in neutrals:
        zero_ref=free_energy(pKas[neutrals.index(acid)], charges[neutrals.index(acid)])
    else:
        zero_ref=0
    return zero_ref

def load_pmf_batches(acid, pmf_dir):
    """
    Load PMF from 3 trajectory files (each with 3 batch columns),
    compute mean + SE over 9 batches. PMF is symmetrized (z >= 0).
    """
    batch_data = []
    for traj in [1, 2, 3]:
        filepath = os.path.join(pmf_dir, acid.lower(), f'trajectory{traj}.dat')
        if not os.path.isfile(filepath):
            continue
        df = pd.read_csv(filepath, sep=r'\s+')
        z_col = df.columns[0]
        batch_cols = df.columns[1:]

        for col in batch_cols:
            df_batch = df[[z_col, col]].copy()
            df_batch.columns = ['z', 'pmf']
            df_batch = df_batch.dropna()
            batch_data.append(df_batch)

    if len(batch_data) == 0:
        return None

    # Merge all batches on z
    result = batch_data[0][['z']].copy()
    for i, bd in enumerate(batch_data):
        result = pd.merge(result, bd, on='z', how='outer', suffixes=('', f'_{i}'))
        if i == 0:
            result = result.rename(columns={'pmf': f'pmf_{i}'})

    pmf_cols = [c for c in result.columns if c.startswith('pmf')]
    result['PMF_mean'] = result[pmf_cols].mean(axis=1)
    result['std_error'] = result[pmf_cols].std(axis=1, ddof=1) / np.sqrt(len(pmf_cols))
    return result[['z', 'PMF_mean', 'std_error']].dropna().sort_values('z')


def plot_pmf_mono_vs_total_45A_grid(acid_list, label_list, name, titrable=False,
                                     ncols=4, x_grid_interval=10):
    """
    Grid plot comparing PMF monomer (4.5A) vs PMF total. No percentage scaling.
    """
    n_acids = len(acid_list)
    nrows = int(np.ceil(n_acids / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 2.2 * nrows),
                              gridspec_kw={'hspace': 0.3, 'wspace': 0.25}, sharex=True)
    axes_flat = axes.flatten()

    for idx, (acid, label) in enumerate(zip(acid_list, label_list)):
        ax = axes_flat[idx]

        data_total = load_pmf_batches(acid, PMF_TOTAL)
        data_mono = load_pmf_batches(acid, PMF_MONO)

        if titrable:
            correction = get_ref_per_acid(acid)
        else:
            correction = 0

        if data_total is not None:
            x_t, y_t, e_t = data_total['z'], data_total['PMF_mean'] + correction, data_total['std_error']
            ax.fill_between(x_t, y_t - e_t, y_t + e_t, alpha=0.3, color='purple')
            ax.plot(x_t, y_t, label='Full', lw=1.5, color='purple')
            #print(e_t)

        if data_mono is not None:
            x_m, y_m, e_m = data_mono['z'], data_mono['PMF_mean'] + correction, data_mono['std_error']
            ax.fill_between(x_m, y_m - e_m, y_m + e_m, alpha=0.3, color='green')
            ax.plot(x_m, y_m, label='Mono', lw=1.5, color='green', linestyle='--')
            #print(e_m)

        ax.set_xlim(0, 35)
        ax.set_title(label, fontsize=10, fontweight='bold')
        ax.xaxis.set_major_locator(ticker.MultipleLocator(x_grid_interval))
        ax.grid(True, which='major', linestyle='--', alpha=0.5)
        ax.tick_params(axis='both', labelsize=8)

        if idx == 0:
            ax.legend(fontsize=10)

    for idx in range(n_acids, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.text(0.5, 0.03, "z (Å)", ha='center', fontsize=11)
    fig.text(0.07, 0.5, "PMF (kJ/mol)", va='center', rotation='vertical', fontsize=11)

    plt.tight_layout(rect=[0.03, 0.03, 1, 1])
    plt.savefig(save_dir + f'{name}.png', bbox_inches='tight', dpi=600)
    plt.show()


# Hydrophobic subset
aa_hydrophobic = ['SCV', 'SCL', 'SCI', 'SCM', 'SCF', 'SCY', 'SCW', 'SCRN']
labels_hydrophobic = ['VAL', 'LEU', 'ILE', 'MET', 'PHE', 'TYR$^0$', 'TRP', 'ARG$^0$']

plot_pmf_mono_vs_total_45A_grid(aa_hydrophobic, labels_hydrophobic,
                                 'pmf_mono_vs_total_hydrophobic', titrable=True)

In [ ]:
##############################################################################
# Mono vs Multi distribution plot (using monomer_4.5A and multimer_4.5A data)
# Scaled by monomer/multimer percentages from the previous cell
##############################################################################
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

DIST_MONO = '../data/distribution_data/monomer_4.5A'
DIST_MULTI = '../data/distribution_data/multimer_4.5A'
save_dir = '../plot/SUPP_monomer/'

def load_distribution_batches(acid, dist_dir, pad_to=None):
    """
    Load distribution data from 3 trajectory files (each with 3 batch columns),
    symmetrize each batch, and return mean + SE over 9 batches.
    If pad_to is set, extend z range to that value with zeros (using 1.0 spacing).
    """
    batch_syms = []
    for traj in [1, 2, 3]:
        filepath = os.path.join(dist_dir, acid.lower(), f'trajectory{traj}.dat')
        if not os.path.isfile(filepath):
            continue
        df = pd.read_csv(filepath, sep=r'\s+')
        z_col = df.columns[0]
        batch_cols = df.columns[1:]  # 3 batch columns

        for col in batch_cols:
            df_batch = df[[z_col, col]].copy()
            df_batch.columns = ['z', 'density']
            df_batch = df_batch.dropna()

            # Symmetrize
            df_pos = df_batch[df_batch['z'] >= 0].copy().sort_values('z').reset_index(drop=True)
            df_neg = df_batch[df_batch['z'] < 0].copy()
            df_neg['z'] = -df_neg['z']
            df_neg = df_neg.sort_values('z').reset_index(drop=True)
            merged = pd.merge(df_pos, df_neg, on='z', suffixes=('_pos', '_neg'), how='outer')
            merged['density'] = merged[['density_pos', 'density_neg']].mean(axis=1)
            batch_syms.append(merged[['z', 'density']].dropna())

    if len(batch_syms) == 0:
        return None

    # Merge all batches on z
    result = batch_syms[0][['z']].copy()
    for i, bs in enumerate(batch_syms):
        result = pd.merge(result, bs, on='z', how='outer', suffixes=('', f'_{i}'))
        if i == 0:
            result = result.rename(columns={'density': f'density_{i}'})

    dens_cols = [c for c in result.columns if c.startswith('density')]

    # Pad to z=pad_to with zeros if needed
    if pad_to is not None:
        z_max = result['z'].max()
        if z_max < pad_to:
            dz = 1.0
            z_pad = np.arange(z_max + dz, pad_to + dz / 2, dz)
            pad_df = pd.DataFrame({'z': z_pad})
            for col in dens_cols:
                pad_df[col] = 0.0
            result = pd.concat([result, pad_df], ignore_index=True)

    result['mean'] = result[dens_cols].mean(axis=1)
    result['se'] = result[dens_cols].std(axis=1, ddof=1) / np.sqrt(len(dens_cols))
    return result[['z', 'mean', 'se']].dropna().sort_values('z')


def plot_mono_vs_multi_45A_grid(acid_list, label_list, name, scale=True, ncols=4, x_grid_interval=10):
    """
    Grid plot comparing monomer vs multimer distributions (4.5A cutoff).
    Distributions are scaled by the monomer/multimer percentages.
    """
    n_acids = len(acid_list)
    nrows = int(np.ceil(n_acids / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 2.2 * nrows),
                              gridspec_kw={'hspace': 0.3, 'wspace': 0.25}, sharex=True)
    axes_flat = axes.flatten()

    for idx, (acid, label) in enumerate(zip(acid_list, label_list)):
        ax = axes_flat[idx]

        data_mono = load_distribution_batches(acid, DIST_MONO)
        data_multi = load_distribution_batches(acid, DIST_MULTI, pad_to=50)

        # Get correction factors from monomer_rates dict
        if scale and acid in monomer_rates:
            mono_factor = monomer_rates[acid] / 100.0
            multi_factor = 1.0 - mono_factor
        else:
            mono_factor = 1.0
            multi_factor = 1.0

        if data_mono is not None:
            ax.fill_between(data_mono['z'],
                           (data_mono['mean'] - data_mono['se']) * mono_factor * 100,
                           (data_mono['mean'] + data_mono['se']) * mono_factor * 100,
                           alpha=0.3, color='tab:blue')
            ax.plot(data_mono['z'], data_mono['mean'] * mono_factor * 100,
                   label='Mono', lw=1.5, color='tab:blue')
            #print(data_mono['se'] * mono_factor * 100)

        if data_multi is not None:
            ax.fill_between(data_multi['z'],
                           (data_multi['mean'] - data_multi['se']) * multi_factor * 100,
                           (data_multi['mean'] + data_multi['se']) * multi_factor * 100,
                           alpha=0.3, color='tab:orange')
            ax.plot(data_multi['z'], data_multi['mean'] * multi_factor * 100,
                   label='Multi', lw=1.5, color='tab:orange', linestyle='--')
            #print(data_multi['se'] * multi_factor * 100)
        ax.set_xlim(0, 35)
        ax.set_title(label, fontsize=10, fontweight='bold')
        ax.xaxis.set_major_locator(ticker.MultipleLocator(x_grid_interval))
        ax.grid(True, which='major', linestyle='--', alpha=0.5)
        ax.tick_params(axis='both', labelsize=8)

        if idx == 0:
            ax.legend(fontsize=10)

    for idx in range(n_acids, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.text(0.5, 0.03, "z (Å)", ha='center', fontsize=11)
    fig.text(0.07, 0.5, "$g_{norm}(z)$ (%)", va='center', rotation='vertical', fontsize=11)

    plt.tight_layout(rect=[0.03, 0.03, 1, 1])
    plt.savefig(save_dir + f'{name}.png', bbox_inches='tight', dpi=600)
    plt.show()


# Hydrophobic subset
aa_hydrophobic = ['SCV', 'SCL', 'SCI', 'SCM', 'SCF', 'SCY', 'SCW', 'SCRN']
labels_hydrophobic = ['VAL', 'LEU', 'ILE', 'MET', 'PHE', 'TYR$^0$', 'TRP', 'ARG$^0$']

plot_mono_vs_multi_45A_grid(aa_hydrophobic, labels_hydrophobic,
                            'mono_vs_multi_distribution_hydrophobic', scale=True)